# BEFORE THE BATTERY DIES: Forecasting Lithium-Ion Battery Degradation Using Statistical Time-Series Models

**Academic Context:** Undergraduate Time Series Analysis (TSA) Lab AAT Project  
**Domain:** Industrial / Energy Systems (Battery Health Monitoring & Prognostics)  
**Central Research Question:** *Can trend-aware statistical forecasting models improve the prediction of lithium-ion battery capacity degradation compared with conventional ARIMA forecasting?*  
**Dataset:** EV Lithium Ion Battery State of Health Dataset (Mendeley Data V2, DOI: `10.17632/7ctkcfnsdw.2`) / NASA Ames PCoE Cell **B0006** ($24^\circ\text{C}$ ambient temperature, 168 discharge cycles across 614 experimental operations).

---

## Core Theoretical Concepts & Four Compared Models

1. **Cycle-Indexed Time Series:** Battery degradation occurs over sequential charge–discharge cycles $t \in \{1, 2, \dots, N\}$ rather than calendar timestamps:
   $$Y = \{Q_1, Q_2, \dots, Q_N\}, \quad \hat{Q}_{n+h} = f(Q_1, Q_2, \dots, Q_n)$$
2. **State of Health ($\text{SOH}$):**
   $$\text{SOH}_t (\%) = \frac{Q_t}{Q_{\text{initial}}} \times 100, \quad Q_{\text{initial}} = Q_1 = 2.051974\text{ Ah}$$
3. **Model 1 — Baseline $\text{ARIMA}(0, 1, 1)$ without Drift (`trend='n'`):** Models first-differenced increments $\Delta Q_t = \varepsilon_t + \theta_1 \varepsilon_{t-1}$ with zero unconditional drift.
4. **Model 2 — $\text{ARIMA}(0, 1, 1)$ with Linear Drift (`trend='t'`):** Models $\Delta Q_t = c + \varepsilon_t + \theta_1 \varepsilon_{t-1}$ with deterministic per-cycle fade slope $c < 0$.
5. **Model 3 — Holt's Linear Trend:** Explicitly smooths both capacity level ($L_t$) and per-cycle degradation slope ($T_t$):
   $$L_t = \alpha Q_t + (1 - \alpha)(L_{t-1} + T_{t-1}), \quad T_t = \beta(L_t - L_{t-1}) + (1 - \beta)T_{t-1}, \quad \hat{Q}_{t+h|t} = L_t + h T_t$$
6. **Model 4 — Damped Holt's Trend:** Introduces autoregressive damping $0 < \phi < 1$ on the slope state ($\hat{Q}_{t+h|t} = L_t + \sum_{j=1}^h \phi^j T_t$).


In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import run_full_data_audit
from src.preprocessing import preprocess_battery_data
from src.eda import generate_eda_figures
from src.stationarity import analyze_stationarity
from src.models import (
    chronological_train_test_split,
    select_and_fit_arima,
    select_and_fit_holt,
    run_rolling_origin_evaluation,
    run_rolling_one_step_recalibration,
    estimate_rul_threshold_crossing,
)
from src.evaluation import evaluate_all_models

print("Project Root:", PROJECT_ROOT)

Project Root: C:\Users\User1\.gemini\antigravity\scratch\before-the-battery-dies


---
## Phase 1 — Raw Data Inspection & Methodology Audit

We inspect all three raw Excel files in `data/raw/` (`Discharge AH.xlsx`, `Discharge Summary.xlsx`, and `SoH Data.xlsx`) and enforce the methodological audit fixes documented in `reports/methodology_audit.md`:
- **Training-Only ACF/PACF (Figures 5 & 6):** Computed strictly on Cycles 1–134 to prevent test-set leakage into Box–Jenkins order selection.
- **Fair Separation of Evaluation Regimes:** Training (1-step in-sample), Inner Validation (Cycles 111–134, 24-step), Held-Out Test (Cycles 135–168, 34-step), and Online 1-Step Recalibration are reported in dedicated tables.
- **Causal Rolling-Origin & RUL Tuning:** At every rolling cutoff (`90, 105, 120, 134, 148`) and RUL origin (`Cycle 60`), smoothing parameters are tuned strictly on past cycles.

In [2]:
raw_data, audit_results = run_full_data_audit(PROJECT_ROOT / "data" / "raw")
for fname, sheets in audit_results["files"].items():
    for sname, info in sheets.items():
        print(f"File: {fname} | Sheet: {sname} | Shape: ({info['rows']}, {info['cols']}) | Missing: {info['total_missing']} | Duplicates: {info['duplicate_rows']}")
        print("  Columns:", [c["column"] for c in info["columns"]])
print("\nCross-check Summary:", audit_results["cross_checks"])

File: Discharge AH.xlsx | Sheet: Sheet1 | Shape: (50285, 9) | Missing: 1 | Duplicates: 0
  Columns: ['Cycle_Number', 'Cycle_Type', 'Ambient_Temperature', 'Time', 'Current_measured', 'Voltage_measured', 'Current Mod', 'delta t', 'Discharge Ah']
File: Discharge Summary.xlsx | Sheet: Sheet1 | Shape: (168, 2) | Missing: 0 | Duplicates: 0
  Columns: ['Dis-ch Cycle', 'Dis-chg Ah']
File: SoH Data.xlsx | Sheet: Sheet1 | Shape: (168, 4) | Missing: 0 | Duplicates: 0
  Columns: ['Charge Cycle', 'Cycle #', 'Dis-chg Ah', 'SoH']

Cross-check Summary: {"unique_discharge_cycles": 168, "operation_cycle_min": 2, "operation_cycle_max": 614, "ambient_temperatures_c": [24], "initial_capacity_ah": 2.051974, "final_capacity_ah": 1.203769, "min_capacity_ah": 1.172108}


---
## Phase 2 — Preprocessing & Derived Health Indicators

We construct the clean cycle-indexed series (`battery_id = "NASA_B0006"`, `cycle_number = 1..168`), derive `soh_percent` and `capacity_fade_ah`, merge cycle-level telemetry from `Discharge AH.xlsx`, and save `data/processed/battery_capacity.csv`.

In [3]:
clean_df = preprocess_battery_data(
    raw_data,
    output_csv_path=PROJECT_ROOT / "data" / "processed" / "battery_capacity.csv"
)
print("Clean dataset shape:", clean_df.shape)
display(clean_df.head(10))

Clean dataset shape: (168, 14)


---
## Phase 3 — Exploratory Data Analysis (EDA) & Plot Interpretations

We generate all 8 EDA figures (`results/figures/fig01` through `fig08`) with consistent axis labels (`Discharge Cycle` and `Discharge Capacity (Ah)`), computing Figures 5 & 6 (ACF & PACF) strictly on the chronological training set (Cycles 1–134).

In [4]:
eda_res = generate_eda_figures(clean_df, PROJECT_ROOT / "results" / "figures", train_end_cycle=134)
for k, v in eda_res["summary_stats"].items():
    print(f"{k}: {v}")

n_cycles: 168
q_initial_ah: 2.051974
q_final_ah: 1.20376877018717
q_min_ah: 1.1721076019223848
q_min_cycle: 164
q_mean_ah: 1.5594619774006687
q_std_ah: 0.2506410164488055
total_fade_ah: 0.84820522981283
total_fade_percent: 41.33606310350972
positive_regen_cycles_count: 31
max_regen_jump_ah: 0.1518125164628718
max_regen_cycle: 90
first_cycle_below_80pct_soh: 61
first_cycle_below_70pct_soh: 103
first_cycle_below_1_40ah: 112
ambient_temp_unique_c: [24]


---
## Phase 4 — Stationarity Analysis (ADF Test & Differencing Justification)

We perform the Augmented Dickey–Fuller (ADF) test on both the training sequence (Cycles 1–134) and the full series before and after first-order differencing $\Delta Q_t = Q_t - Q_{t-1}$.

In [5]:
train_df, test_df = chronological_train_test_split(clean_df, train_ratio=0.80)
stat_res = analyze_stationarity(clean_df, train_end_idx=len(train_df))
for key in ["adf_train_level", "adf_train_diff1", "adf_full_level", "adf_full_diff1"]:
    r = stat_res[key]
    print(f"{r['series_name']}: ADF={r['adf_statistic']:.4f}, p={r['p_value']:.4e}, Crit(5%)={r['critical_values']['5%']:.4f} -> {r['conclusion']}")
print("Recommended d:", stat_res["recommended_d"])

Training Series Q_t (Level, d=0): ADF=-1.3520, p=6.0507e-01, Crit(5%)=-2.8840 -> Fail to reject H0 at alpha=0.05: Series 'Training Series Q_t (Level, d=0)' is NON-STATIONARY (unit root present).
Training Series ΔQ_t (First Difference, d=1): ADF=-8.5062, p=1.2013e-13, Crit(5%)=-2.8840 -> Reject H0 at alpha=0.05: Series 'Training Series ΔQ_t (First Difference, d=1)' is STATIONARY (no unit root).
Full Series Q_t (Level, d=0): ADF=-1.3659, p=5.9850e-01, Crit(5%)=-2.8793 -> Fail to reject H0 at alpha=0.05: Series 'Full Series Q_t (Level, d=0)' is NON-STATIONARY (unit root present).
Full Series ΔQ_t (First Difference, d=1): ADF=-9.5790, p=2.1834e-16, Crit(5%)=-2.8793 -> Reject H0 at alpha=0.05: Series 'Full Series ΔQ_t (First Difference, d=1)' is STATIONARY (no unit root).
Recommended d: 1


---
## Phases 5, 6, 7 & 9 — Fair Model Comparison, Horizon Analysis, Rolling-Origin Cross-Validation, Residual Diagnostics & RUL Estimation

Using the identical chronological split (Training Cycles 1–134, Inner Validation Cycles 111–134, Held-Out Test Cycles 135–168, $H = 34$ cycles), we evaluate:
- **Model 1:** $\text{ARIMA}(0, 1, 1)$ Baseline (`trend='n'`)
- **Model 2:** $\text{ARIMA}(0, 1, 1)$ with Linear Drift (`trend='t'`)
- **Model 3:** Holt's Linear Trend (Val-Tuned $\alpha=0.10, \beta=0.05$ and Train-MLE $\alpha=0.88, \beta=0.00$)
- **Model 4:** Damped Holt's Trend (Val-Tuned $\alpha=0.60, \beta=0.05, \phi=0.98$)

In [6]:
arima_res = select_and_fit_arima(train_df["discharge_capacity_ah"], test_horizon=len(test_df), d=1)
holt_res = select_and_fit_holt(train_df["discharge_capacity_ah"], test_horizon=len(test_df))
rolling_origin_df = run_rolling_origin_evaluation(clean_df, arima_order=arima_res["best_order"])
rolling_1step_res = run_rolling_one_step_recalibration(
    train_df["discharge_capacity_ah"], test_df["discharge_capacity_ah"],
    arima_order=arima_res["best_order"],
    holt_alpha=holt_res["smoothing_level_alpha"], holt_beta=holt_res["smoothing_trend_beta"]
)
rul_res = estimate_rul_threshold_crossing(
    clean_df, origin_cycle=60, max_horizon=108, eol_capacity_ah=1.40,
    arima_order=arima_res["best_order"]
)
eval_res = evaluate_all_models(
    train_df, test_df, arima_res, holt_res,
    rolling_1step_res, rolling_origin_df, rul_res, PROJECT_ROOT / "results"
)

print("=== HELD-OUT MULTI-STEP TEST METRICS (CYCLES 135-168) ===")
print(eval_res["comparison_df"].to_string(index=False))
print("\n=== SEPARATED TRAIN / VALIDATION / TEST METRICS ===")
print(eval_res["split_metrics_df"].to_string(index=False))
print("\n=== ONLINE 1-STEP RECALIBRATION METRICS ===")
print(eval_res["online_df"].to_string(index=False))
print("\n=== FORECAST HORIZON ANALYSIS (h = 5, 10, 20, 34) ===")
print(eval_res["horizon_comparison_df"].to_string(index=False))
print("\n=== LEAKAGE-FREE ROLLING-ORIGIN SUMMARY (MEAN +/- SD ACROSS 5 ORIGINS) ===")
print(eval_res["rolling_origin_summary_df"].to_string(index=False))
print("\n=== RESIDUAL DIAGNOSTICS SUMMARY ===")
print(eval_res["residual_diagnostics_df"].to_string(index=False))
print("\n=== CAUSAL RUL ANALYSIS (ORIGIN CYCLE 60, EOL = 1.40 Ah) ===")
print(json.dumps(eval_res["summary"]["rul_analysis"], indent=2))

=== HELD-OUT MULTI-STEP TEST METRICS (CYCLES 135-168) ===
Model,Evaluation_Set,Forecast_Mode,Horizon_Cycles,MAE_Ah,RMSE_Ah,MAPE_Percent,sMAPE_Percent,Val_RMSE_Ah,Train_RMSE_Ah
"Model 1: ARIMA(0, 1, 1) — Baseline",Held-Out Test (Cycles 135-168),Multi-Step (34-Cycle Horizon),34,0.10093964984308122,0.11152800446697082,8.203385050554324,7.805252695093507,0.04286081159710082,0.025599319454899544
"Model 2: ARIMA(0, 1, 1) with Drift",Held-Out Test (Cycles 135-168),Multi-Step (34-Cycle Horizon),34,0.012977070503773273,0.01521501121138421,1.0298799229409252,1.0278320360456121,0.046890490151565,0.02495742821802294
"Model 3: Holt's Linear Trend (Val-Tuned: alpha=0.10, beta=0.05)",Held-Out Test (Cycles 135-168),Multi-Step (34-Cycle Horizon),34,0.02252878829408079,0.027394250253891746,1.8407727255362,1.8166848770530146,0.017316697955264408,0.035327140581664335
"Model 3b: Holt's Linear Trend (Train-MLE: alpha=0.88, beta=0.00)",Held-Out Test (Cycles 135-168),Multi-Step (34-Cycle Horizon),34,0.0128772